# Baseline — Qwen2.5-VL-3B (zero-shot) on ScreenSpot-V2

Smoke test for the shared infrastructure. Run this once before any task notebook.

**Hypothesis to confirm:** unadapted Qwen2.5-VL-3B is around 27% on ScreenSpot-V2 (per the proposal, §3). If this notebook reports something within ±5 pp of that, the eval harness is wired correctly.

In [ ]:
# Colab bootstrap. On a local Jupyter, this is a no-op since the env is already set up.
import sys
from pathlib import Path

REPO_URL = "https://github.com/ali-epita/action-learning-ais5.git"
REPO_DIR = "/content/action-learning-ais5"

if "google.colab" in sys.modules:
    if not Path(REPO_DIR).exists():
        !git clone --depth 1 {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

    # --no-deps keeps Colab's pre-bundled torch + CUDA build untouched.
    !pip install -q -e . --no-deps

    # Pin transformers <5: the Qwen2.5-VL wrapper was written against 4.49+,
    # and Colab currently ships transformers 5.0 which renamed several classes.
    # Strip Colab's pre-installed torchao (peft 0.17+ raises ImportError on the
    # 0.10.0 that ships with Colab instead of treating it as absent). LoRA
    # fine-tuning here does not depend on torchao.
    !pip uninstall -y -q torchao
    !pip install -q "transformers>=4.49,<5" "accelerate>=0.34" "datasets>=3.0" \
                    "peft>=0.13" "qwen-vl-utils>=0.0.10" \
                    rich tqdm pyyaml typer pillow huggingface_hub safetensors

    # pip's editable install drops a .pth file in site-packages but the current
    # kernel doesn't always rescan, so put src/ on sys.path explicitly.
    src_dir = str(Path(REPO_DIR) / "src")
    if src_dir not in sys.path:
        sys.path.insert(0, src_dir)
    import importlib
    importlib.invalidate_caches()

import ais5
print(f"ais5 v{ais5.__version__} ready  (colab={'google.colab' in sys.modules})")

In [2]:
from ais5.data import load_benchmark
from ais5.eval import evaluate_model, by_target_size, by_ui_type
from ais5.models import get_model
from ais5.utils import set_global_seed, setup_logging

setup_logging()
set_global_seed(42)

In [ ]:
model = get_model('qwen2.5-vl-3b')
samples = list(load_benchmark('screenspot-v2'))
print(f'{len(samples)} samples loaded')
print(f'first sample: {samples[0]}')

[05/06/26 14:20:37] INFO     HTTP Request: HEAD                                                                    
                             https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct/resolve/main/config.json "HTTP/1.1 
                             307 Temporary Redirect"

                    WARNING  Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN
                             to enable higher rate limits and faster downloads.

                    INFO     HTTP Request: HEAD                                                                    
                             https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-VL-3B-Instruct/66285546d2
                             b821cf421d4f5eb2576359d3770cd3/config.json "HTTP/1.1 200 OK"

                    INFO     HTTP Request: GET                                                                     
                             https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-VL-3B-Instruct/66285546d2
                             b821cf421d4f5eb2576359d3770cd3/config.json "HTTP/1.1 200 OK"

config.json: 0.00B [00:00, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

[05/06/26 14:20:39] INFO     HTTP Request: HEAD                                                                    
                             https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct/resolve/66285546d2b821cf421d4f5eb25
                             76359d3770cd3/model-00001-of-00002.safetensors "HTTP/1.1 302 Found"

                    INFO     HTTP Request: HEAD                                                                    
                             https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct/resolve/66285546d2b821cf421d4f5eb25
                             76359d3770cd3/model-00002-of-00002.safetensors "HTTP/1.1 302 Found"

                    INFO     HTTP Request: GET                                                                     
                             https://huggingface.co/api/models/Qwen/Qwen2.5-VL-3B-Instruct/xet-read-token/66285546d
                             2b821cf421d4f5eb2576359d3770cd3 "HTTP/1.1 200 OK"

                    INFO     HTTP Request: GET                                                                     
                             https://huggingface.co/api/models/Qwen/Qwen2.5-VL-3B-Instruct/xet-read-token/66285546d
                             2b821cf421d4f5eb2576359d3770cd3 "HTTP/1.1 200 OK"

In [ ]:
# Limit to 50 for a quick sanity run; remove `limit=` for the full eval.
run = evaluate_model(model, samples, benchmark='screenspot-v2', limit=50)
print(f'accuracy = {run.accuracy:.4f} (n={len(run.results)})')
print(f'avg latency = {run.avg_latency_ms:.1f} ms' if run.avg_latency_ms else '')

In [ ]:
# Parser hit-rate by dialect. Reruns `parse_click` on each raw response so we
# see which pattern fired (the runner doesn't carry that through to ClickResult).
# A healthy run is >=90% qwen-click — the dialect we explicitly ask for.
from collections import Counter

from ais5.prompt.action import parse_click

samples_by_id = {s.sample_id: s for s in samples}
parser_counts: Counter[str] = Counter()
for r in run.results:
    sample = samples_by_id.get(r.sample_id)
    if sample is None:
        continue
    reparsed = parse_click(r.raw_response, image_size=sample.image_size)
    parser_counts[reparsed.parser] += 1

n = len(run.results)
hit = n - parser_counts["none"]
print(f"parser hit rate: {hit / n:.1%}  (n={n})")
for name, c in parser_counts.most_common():
    print(f"  {name:14s} {c:3d} / {n} ({c/n:.1%})")

print("\nfirst 3 wrong-answer raw responses:")
for r in [r for r in run.results if not r.correct][:3]:
    print(f"  sample={r.sample_id!r}  pred={r.pred}  bbox={r.bbox}")
    print(f"  raw: {r.raw_response[:200].strip()}")
    print()

In [ ]:
# Wrong-answer overlays. Gold bbox = lime outline; predicted click = red x.
# Near-miss inside or just outside the box => resolution / small-target issue.
# Far-miss across the screen => prompt or parser issue, not a knowledge gap.
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

wrong = [r for r in run.results if not r.correct][:5]
if not wrong:
    print("(no wrong samples in this smoke run)")
else:
    fig, axes = plt.subplots(1, len(wrong), figsize=(4 * len(wrong), 4))
    if len(wrong) == 1:
        axes = [axes]
    for ax, r in zip(axes, wrong):
        sample = samples_by_id[r.sample_id]
        ax.imshow(sample.image)
        x1, y1, x2, y2 = r.bbox
        ax.add_patch(mpatches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2, edgecolor="lime", facecolor="none",
        ))
        if r.pred is not None:
            ax.plot(r.pred[0], r.pred[1], "rx", markersize=15, mew=3)
        ax.set_title(f"{(sample.sample_id or '')[:20]}\n{sample.instruction[:50]}", fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
by_target_size(run.results)

In [ ]:
by_ui_type(run.results)

In [ ]:
# Persist the smoke run as JSON so we don't have to rerun it if the kernel dies
# before the full eval finishes.
import json
from pathlib import Path

results_dir = Path("results/eval")
results_dir.mkdir(parents=True, exist_ok=True)
smoke_file = results_dir / f"screenspot-v2__{model.name}__smoke_n{len(run.results)}.json"

payload = {
    "model": model.name,
    "benchmark": "screenspot-v2",
    "accuracy": run.accuracy,
    "n_samples": len(run.results),
    "avg_latency_ms": run.avg_latency_ms,
    "parser_counts": dict(parser_counts),
    "results": [
        {
            "sample_id": r.sample_id,
            "pred": list(r.pred) if r.pred else None,
            "bbox": list(r.bbox),
            "correct": r.correct,
            "target_type": r.target_type,
            "ui_type": r.ui_type,
            "target_relative_area": r.target_relative_area,
            "latency_ms": r.latency_ms,
            "raw_response": r.raw_response,
        }
        for r in run.results
    ],
}
smoke_file.write_text(json.dumps(payload, indent=2))
print(f"saved smoke results -> {smoke_file}")

## Sanity checks

- Did the parser succeed on most samples? Look for `<click>` patterns in `run.results[i].raw_response`.
- Are the wrong-answer points falling near the bbox or far away? Far → prompt issue. Near → resolution issue.
- Is the average latency reasonable (~1-3 s on Colab T4)? Much higher → check `device_map`.

In [ ]:
# Full ScreenSpot-V2 baseline (1,272 samples). Flip RUN_FULL = True when the
# smoke run above looks healthy (>=90% parser hit rate, latency ~1-3 s/sample
# on A100). Writes one JSON line per sample to results/eval/...full.jsonl, so
# a Colab disconnect doesn't lose what already ran.
RUN_FULL = False

if RUN_FULL:
    from ais5.eval.click import point_in_bbox

    jsonl_path = Path("results/eval") / f"screenspot-v2__{model.name}__full.jsonl"
    jsonl_path.unlink(missing_ok=True)

    def _write_line(sample, out, dt_ms):
        point = out.parsed.point
        correct = point is not None and point_in_bbox(point, sample.bbox)
        line = {
            "sample_id": sample.sample_id,
            "instruction": sample.instruction,
            "pred": list(point) if point else None,
            "bbox": list(sample.bbox),
            "correct": bool(correct),
            "target_type": sample.target_type,
            "ui_type": sample.ui_type,
            "target_relative_area": sample.target_relative_area,
            "latency_ms": dt_ms,
            "raw_response": out.text,
            "parser": out.parsed.parser,
        }
        with jsonl_path.open("a") as f:
            f.write(json.dumps(line) + "\n")

    full_run = evaluate_model(
        model, samples, benchmark="screenspot-v2", on_predict=_write_line,
    )
    print(f"accuracy = {full_run.accuracy:.4f}  (n={len(full_run.results)})")
    if full_run.avg_latency_ms is not None:
        print(f"avg latency = {full_run.avg_latency_ms:.1f} ms")
    print(f"-> {jsonl_path}")
else:
    print("Smoke healthy? Flip RUN_FULL = True and re-run this cell.")